# Historical Backtest: ETH/USDC 30bps Fee Concentration Insurance

**Question:** Would PLPs have been better off with ThetaSwap insurance?

**Method:** Replay 600 real positions (41 days, 2025-12-05 to 2026-01-14) through the hybrid insurance mechanism (main.pdf §6). Compare actual P&L vs counterfactual hedged P&L at each γ.

**Insurance mechanism:**
- Premium: PLP pays γ% of fees into reserve R at exit
- Trigger: Δ⁺ > Δ* = 0.09 (econometric turning point)  
- Payout: D = (Δ⁺ − Δ*)/(1 − Δ*) · R via donate(), pro-rata to insured positions

In [ ]:
from econometrics.data import DAILY_AT_MAP, DAILY_AT_NULL_MAP, IL_MAP, RAW_POSITIONS
from backtest.daily import build_daily_states
from backtest.sweep import run_single_backtest, run_gamma_sweep
from backtest.calibrate import derive_gamma_star, compute_avg_fees
from backtest.plotting import money_plot, reserve_plot, hedge_distribution_plot

POOL_DAILY_FEE = 50_000.0
DELTA_STAR = 0.09

# Convert tuples (burn_date, blocklife, fee_usd) to dicts expected by backtest
positions = [{"burn_date": bd, "blocklife": bl} for bd, bl, _ in RAW_POSITIONS]

daily_states = build_daily_states(
    DAILY_AT_MAP, DAILY_AT_NULL_MAP, IL_MAP, positions,
    pool_daily_fee=POOL_DAILY_FEE,
)
print(f"Days: {len(daily_states)}")
print(f"Positions: {sum(1 for p in positions if p['blocklife'] > 1)}")

## 1. Trigger Days

In [ ]:
import matplotlib.pyplot as plt

days = [s.day[-5:] for s in daily_states]
deltas = [s.delta_plus for s in daily_states]
triggers = [s.delta_plus > DELTA_STAR for s in daily_states]

print(f"Trigger days: {sum(triggers)} / {len(daily_states)}")
print(f"Days with Δ⁺ > 0: {sum(1 for d in deltas if d > 0)} / {len(daily_states)}")
print()
for s in daily_states:
    if s.delta_plus > DELTA_STAR:
        print(f"  {s.day}: Δ⁺ = {s.delta_plus:.4f}")

fig, ax = plt.subplots(figsize=(12, 3))
colors = ["#e74c3c" if t else "#3498db" for t in triggers]
ax.bar(range(len(days)), deltas, color=colors, edgecolor="black", linewidth=0.3)
ax.axhline(y=DELTA_STAR, color="red", linestyle="--", label=f"Δ* = {DELTA_STAR}")
ax.set_ylabel("Δ⁺")
ax.set_title("Daily Concentration Deviation")
ax.set_xticks(range(0, len(days), 5))
ax.set_xticklabels([days[i] for i in range(0, len(days), 5)], rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Gamma Calibration

In [ ]:
base = run_single_backtest(daily_states, positions, gamma=0.001, delta_star=DELTA_STAR)
avg_fees = compute_avg_fees(base.position_pnls)
gamma_star = derive_gamma_star(wtp=110.0, avg_fees=avg_fees)

print(f"Average fees per position: ${avg_fees:,.2f}")
print(f"Econometric WTP: $110")
print(f"Calibrated γ* = {gamma_star:.4f} ({gamma_star:.2%})")

## 3. Gamma Sweep Results

In [ ]:
gammas = sorted(set([0.01, 0.05, 0.10, 0.20, round(gamma_star, 4)]))
results = run_gamma_sweep(daily_states, positions, gammas=gammas, delta_star=DELTA_STAR)

print(f"{'γ':>8} {'Premiums':>12} {'Payouts':>10} {'Triggers':>9} {'Mean HV':>10} "
      f"{'% Better':>9} {'R Peak':>10} {'R Util':>8}")
print("-" * 80)
for r in results:
    star = " ←γ*" if abs(r.gamma - gamma_star) < 0.0001 else ""
    print(f"{r.gamma:>8.2%} {r.total_premiums:>12,.0f} {r.total_payouts:>10,.0f} "
          f"{r.trigger_days:>9} {r.mean_hedge_value:>10,.2f} "
          f"{r.pct_better_off:>8.1%} {r.reserve_peak:>10,.0f} "
          f"{r.reserve_utilization:>8.1%}{star}")

## 4. The Money Plot

In [ ]:
fig = money_plot(results)
plt.show()

## 5. Reserve Dynamics at γ*

In [ ]:
gamma_star_result = [r for r in results if abs(r.gamma - gamma_star) < 0.001][0]
fig = reserve_plot(gamma_star_result)
plt.show()

## 6. Position-Level Hedge Value Distribution

In [ ]:
fig = hedge_distribution_plot(gamma_star_result)
plt.show()

trigger_days_set = {rs.day for rs in gamma_star_result.reserve_states if rs.trigger_fired}
on_trigger = [p for p in gamma_star_result.position_pnls if p.burn_date in trigger_days_set]
off_trigger = [p for p in gamma_star_result.position_pnls if p.burn_date not in trigger_days_set]

print(f"\nPositions exiting on trigger days: {len(on_trigger)}")
if on_trigger:
    print(f"  Mean hedge value: ${sum(p.hedge_value for p in on_trigger)/len(on_trigger):,.2f}")
print(f"\nPositions exiting on non-trigger days: {len(off_trigger)}")
if off_trigger:
    print(f"  Mean hedge value: ${sum(p.hedge_value for p in off_trigger)/len(off_trigger):,.2f}")

## 7. Summary

In [ ]:
print("=== BACKTEST SUMMARY ===\n")
print(f"Pool: ETH/USDC 30bps")
print(f"Window: 2025-12-05 to 2026-01-14 ({len(daily_states)} days)")
print(f"Positions: {len(gamma_star_result.position_pnls)}")
print(f"Trigger days (Δ⁺ > {DELTA_STAR}): {gamma_star_result.trigger_days}")
print(f"Calibrated γ*: {gamma_star:.2%}")
print(f"\nAt γ*:")
print(f"  Total premiums collected: ${gamma_star_result.total_premiums:,.2f}")
print(f"  Total payouts distributed: ${gamma_star_result.total_payouts:,.2f}")
print(f"  Mean hedge value: ${gamma_star_result.mean_hedge_value:,.2f}")
print(f"  Positions better off: {gamma_star_result.pct_better_off:.1%}")
print(f"  Reserve peak: ${gamma_star_result.reserve_peak:,.2f}")
print(f"  Reserve utilization: {gamma_star_result.reserve_utilization:.1%}")